# CLASSIFICATION USING STACKED ENSEMBLING
_Implements a stacked ensembling for classification on an appropriate dataset._

In [1]:
# Imports required modules and methods

import numpy as np

from sklearn.datasets import make_moons

from sklearn.model_selection import cross_val_score, cross_val_predict, train_test_split
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from sklearn.ensemble import VotingClassifier

import matplotlib.pyplot as plt

## The Data
This experiments uses relatively simple binary classification dataset generated synthetically.

In [ ]:
# Generates 2D dataset that makes two interleaving half circles
X_moon, y_moon = make_moons(n_samples=1000, noise=0.30, random_state=42)

# Checks the shape of the generated data
X_moon.shape

In [ ]:
# Visualizes the data to realize the class distribution
plt.scatter(
    X_moon[:,0], X_moon[:,1], 
    c=list(map(lambda x: "red" if x == 0 else "blue", y_moon))  # "red" and "blue" represent class 0 and 1, respectively
)

## Data Preparation

In [ ]:
# Splits the dataset into train and test set with stratification

X_train, X_test, y_train, y_test = train_test_split(
    
    # CODE HERE to pass train set feature values and target (separated by comma), 
    
    test_size=0.2, 
    random_state=42, 
    
    stratify= # CODE HERE to pass train set targets to stratify class distribution across train and test set
    )

## Modeling

### The Estimators

In [ ]:
# Initializes the individual estimators

estimators = [
    ("lr_clf", # CODE HERE),        # INITIALIZE CLASS LogisticRegression WITH `random_state` SET TO 42
    ("rf_clf", # CODE HERE),        # INITIALIZE CLASS LogRandomForestClassifieristicRegression WITH `random_state` SET TO 42
    ("svm_clf", # CODE HERE)        # INITIALIZE CLASS SVC WITH `probability` SET TO True AND `random_state` SET TO 42
]

In [ ]:
# To compare the performance of stacked emsemble, a voting classifier's 
# performance was recorded as reference



voting_clf =    # CODE HERE TO INITIALIZE VOTING CLASSIFIER CLASS VotingClassifier WITH PARAMETERS
                # `estimators` AS ESTIMATORS ALREADY INITIALIZED, `voting` AS "soft" and `n_jobs` as -1 


In [ ]:
# Checks for the CV score of the voting classifier,
print("Voting Classifier's CV Score: {:.3f}".format(
    cross_val_score(voting_clf, X_train, y_train, cv=5, n_jobs=-1).mean()))

In [ ]:
# and then checks for the Test set performance

# CODE HERE TO FIT THE VOTING CLASSIFIER BY CALLING fit() METHOD ON voting_clf PASSING IT TRAIN SET FEATURE VALUES AND TARGETS


print("Voting Classifier's Test Score: {:.3f}".format(voting_clf.score(X_test, y_test)))

### The Blender

The blender gets trained on the cross-validation predictions made by the estimators and with the train set labels as targets. During inference, estimators first predict on the test set and then the blender take these predictions as input and produces the final prediction.

**Preparing Cross-Validation Predictions**

In [ ]:
# Creates space (numpy array) to store CV predictions from all the estimators
estimators_predictions_cv = np.empty(
    shape=(X_train.shape[0], len(estimators)),  # of shape [no. of train examples x no. of estimators 
    dtype=object)

# Verifies the shape of the newly created array to store CV predictions
estimators_predictions_cv.shape

In [61]:
# Enumerates each estimator
for idx, estimator in enumerate(estimators):
    # For each estimator, performs cross validation predictions on train set
    # and stacks predictions horizontally
    estimators_predictions_cv[:, idx] = cross_val_predict(estimator[1], X_train, y_train, method="predict")

In [ ]:
# Shows the few CV predictions for reference
estimators_predictions_cv

**Blender's Cross-validation Performance**

In [ ]:
# Initializes random forest classifier as the blender


blender =   # CODE HERE TO CONSIDER RANDOM FOREST CLASSIFIER AS THE BLENDER BY INITIALIZING ITS CLASS
            # RandomForestClassifier WITH PARAMETERS `n_estimators` SET TO 200, `oob_score` SET TO True,
            # `n_jobs` SET TO -1 AND `random_state` SET TO 42

In [ ]:
# Calculates blender's CV Score (on already generated CV prediction dataset by estimators)
print("Blender's CV Score: {:.3f}".format(
    cross_val_score(blender, estimators_predictions_cv, y_train, cv=5).mean()))

**Blender's Performance on Test Set**

In [ ]:
# First, trains the blender on the CV predictions generated by estimators on train set


# CODE HERE TO FIT THE BLENDER BY CALLING ITS .fit() METHOD PASSING IT ESTIMATORS' CV 
# PREDICTIONS `estimators_predictions_cv` AND TRAIN SET TARGETS

In [ ]:
# Creates space (numpy array) to store predictions from all the estimators on the test set
estimators_predictions_test = np.empty(shape=(X_test.shape[0], len(estimators)), dtype=object)

# Verifies the shape of the newly created array to store test predictions
estimators_predictions_test.shape

In [67]:
# Now, before estimators performs predictions to be input to blender
# for final predictions, they need to be trained first.
for estimator in estimators:
    estimator[1].fit(X_train, y_train)

In [68]:
# Enumerates each estimator
for idx, estimator in enumerate(estimators):
    # For each estimator, performs predictions on the test set
    # and stacks predictions horizontally
    estimators_predictions_test[:, idx] = estimator[1].predict(X_test)

In [ ]:
# Shows the few test predictions for reference
estimators_predictions_test[:10]

In [ ]:
# The above test predictions made by estimators are then input to blender to make final predictions

blender_predictions =   # CODE HERE FOR BLENDER `blender` TO PERFORM PREDICTION ON 
                        # TEST PREDICTIONS `estimators_predictions_test` MADE BY ESTIMATORS
                        # BY CALLING blender.predict() METHOD PASSING IT `estimators_predictions_test`



In [ ]:
# Finally, evaluates the test performance of the blender on predictions made by estimators on the test data
print("Stacked Ensembling Performance: {} [TEST SET]".format(accuracy_score(blender_predictions, y_test)))

**OBSERVATIONS:**

1. What was voting classifier's cross validation score and test score?

2. What was voting classifier's test score?

3. What was blenders cross validation score?

4. What was blenders test score?

5. Analyze the blender's test score compared to that of relatively simpler voting classifier.